In [2]:
"""
=============================================================
PATIENT HEALTHCARE PROJECT — STEP 1: EDA & DATA CLEANING
=============================================================
Author  : Data Analytics Pipeline
Datasets: Patient, Doctor, Visit, Treatment, LabTest
Goal    : Explore, clean, and export analysis-ready CSV files
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
os.makedirs("output_charts", exist_ok=True)
os.makedirs("output_clean", exist_ok=True)

BASE = "Patient Healthcare/Data/"


In [9]:
# ─────────────────────────────────────────────
# 1. LOAD RAW DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("1. LOADING RAW DATA")
print("=" * 60)

patient   = pd.read_excel("Patient.xlsx")
doctor    = pd.read_excel("Doctor.xlsx")
visit     = pd.read_excel("Visit.xlsx")
treatment = pd.read_excel("Treatment.xlsx")
lab       = pd.read_excel("Lab Test.xlsx")

datasets = {"Patient": patient, "Doctor": doctor, "Visit": visit,
            "Treatment": treatment, "LabTest": lab}

for name, df in datasets.items():
    print(f"  {name:12s}: {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols  |  "
          f"nulls={df.isnull().sum().sum():,}")

1. LOADING RAW DATA
  Patient     : 10,000 rows × 23 cols  |  nulls=10,973
  Doctor      :  1,000 rows ×  8 cols  |  nulls=0
  Visit       : 10,000 rows × 11 cols  |  nulls=2,002
  Treatment   : 10,000 rows × 12 cols  |  nulls=0
  LabTest     : 10,000 rows ×  8 cols  |  nulls=0


In [10]:
# ─────────────────────────────────────────────
# 2. DATA CLEANING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. DATA CLEANING")
print("=" * 60)

# ── Patient ──────────────────────────────────
p = patient.copy()

# Drop near-duplicate emergency contact columns
p = p.drop(columns=["Emergency Contact", "Emergency Contact.1"], errors="ignore")

# Fill free-text nulls with 'Unknown'
for col in ["Medical History", "Chronic Conditions", "Allergies"]:
    p[col] = p[col].fillna("None")

# Ensure DateOfBirth is datetime; recalculate Age from it
p["DateOfBirth"] = pd.to_datetime(p["DateOfBirth"], errors="coerce")
today = pd.Timestamp("today")
p["Age_Calc"] = ((today - p["DateOfBirth"]).dt.days / 365.25).round(0).astype("Int64")
p["Age"] = p["Age_Calc"].combine_first(p["Age"].astype("Int64"))
p = p.drop(columns=["Age_Calc"])

# Age groups
bins   = [0, 17, 35, 50, 65, 120]
labels = ["<18", "18–35", "36–50", "51–65", "65+"]
p["Age_Group"] = pd.cut(p["Age"], bins=bins, labels=labels, right=True)

# Full name
p["Full_Name"] = p["First Name"].str.strip() + " " + p["LastName"].str.strip()
p = p.drop_duplicates(subset="Patient ID")
print(f"  Patient   → {len(p):,} rows after dedup")

# ── Doctor ───────────────────────────────────
d = doctor.copy()
d = d.drop_duplicates(subset="Doctor ID")
print(f"  Doctor    → {len(d):,} rows after dedup")

# ── Visit ────────────────────────────────────
v = visit.copy()
v["Visit Date"] = pd.to_datetime(v["Visit Date"], errors="coerce")
v["Visit Year"]  = v["Visit Date"].dt.year
v["Visit Month"] = v["Visit Date"].dt.month
v["Visit MonthName"] = v["Visit Date"].dt.strftime("%b")
v["Prescribed Medications"] = v["Prescribed Medications"].fillna("Not Prescribed")
v = v.drop_duplicates(subset="Visit ID")
print(f"  Visit     → {len(v):,} rows after dedup")

# ── Treatment ────────────────────────────────
t = treatment.copy()
t = t.drop_duplicates(subset="Treatment ID")
print(f"  Treatment → {len(t):,} rows after dedup")

# ── LabTest ──────────────────────────────────
l = lab.copy()
l["Test Date"] = pd.to_datetime(l["Test Date"], errors="coerce")
l = l.drop_duplicates(subset="Lab Result ID")
print(f"  LabTest   → {len(l):,} rows after dedup")



2. DATA CLEANING
  Patient   → 10,000 rows after dedup
  Doctor    → 1,000 rows after dedup
  Visit     → 10,000 rows after dedup
  Treatment → 10,000 rows after dedup
  LabTest   → 10,000 rows after dedup


In [11]:
# ─────────────────────────────────────────────
# 3. MERGE — MASTER TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. BUILDING MASTER TABLE")
print("=" * 60)

master = (
    v
    .merge(p[["Patient ID", "Gender", "Age", "Age_Group", "Blood Type",
              "State", "City", "Insurance Provider", "Chronic Conditions",
              "Allergies", "Full_Name"]], on="Patient ID", how="left")
    .merge(d[["Doctor ID", "Doctor Name", "Specialty",
              "Years Of Experience", "Hospital/Clinic"]], on="Doctor ID", how="left")
    .merge(t[["Visit ID", "Treatment Type", "Treatment Name",
              "Treatment Cost", "Cost", "Status", "Outcome"]], on="Visit ID", how="left")
    .merge(l[["Visit ID", "Test Name", "Test Result",
              "Reference Range"]], on="Visit ID", how="left")
)

print(f"  Master table: {master.shape[0]:,} rows × {master.shape[1]} cols")



3. BUILDING MASTER TABLE
  Master table: 10,000 rows × 37 cols


In [12]:
# ─────────────────────────────────────────────
# 4. EXPLORATORY DATA ANALYSIS  (charts)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. GENERATING EDA CHARTS")
print("=" * 60)


# ── Figure 1 : Patient Demographics ──────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Patient Demographics Overview", fontsize=16, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 4-a  Gender
ax0 = fig.add_subplot(gs[0, 0])
g_cnt = p["Gender"].value_counts()
ax0.pie(g_cnt, labels=g_cnt.index, autopct="%1.1f%%", startangle=90,
        colors=["#5B8DB8", "#D98EA0", "#A8C8A0"])
ax0.set_title("Gender Distribution")

# 4-b  Age Group
ax1 = fig.add_subplot(gs[0, 1])
ag = p["Age_Group"].value_counts().sort_index()
ag.plot(kind="bar", ax=ax1, color="#5B8DB8", edgecolor="white")
ax1.set_title("Age Group Distribution")
ax1.set_xlabel("Age Group"); ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=0)

# 4-c  Blood Type
ax2 = fig.add_subplot(gs[0, 2])
bt = p["Blood Type"].value_counts()
bt.plot(kind="barh", ax=ax2, color="#A8C8A0", edgecolor="white")
ax2.set_title("Blood Type Distribution")
ax2.set_xlabel("Count")

# 4-d  Top 10 States
ax3 = fig.add_subplot(gs[1, :2])
top_states = p["State"].value_counts().head(10)
top_states.plot(kind="bar", ax=ax3, color="#D98EA0", edgecolor="white")
ax3.set_title("Top 10 States by Patient Count")
ax3.set_xlabel("State"); ax3.set_ylabel("Count")
ax3.tick_params(axis="x", rotation=45)

# 4-e  Insurance Provider top 8
ax4 = fig.add_subplot(gs[1, 2])
ins = p["Insurance Provider"].value_counts().head(8)
ins.plot(kind="barh", ax=ax4, color="#B5A0D0", edgecolor="white")
ax4.set_title("Top 8 Insurance Providers")
ax4.set_xlabel("Count")

plt.savefig("output_charts/01_patient_demographics.png", bbox_inches="tight", dpi=150)
plt.close()
print("  ✓ 01_patient_demographics.png")


# ── Figure 2 : Visit Analysis ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Visit Analysis", fontsize=16, fontweight="bold")

# Visit Type
v["Visit Type"].value_counts().plot(kind="bar", ax=axes[0,0], color="#5B8DB8")
axes[0,0].set_title("Visit Types"); axes[0,0].tick_params(axis="x", rotation=25)

# Visit Status
v["Visit Status"].value_counts().plot(kind="pie", ax=axes[0,1], autopct="%1.1f%%",
    colors=["#A8C8A0","#D98EA0","#F5C97A"])
axes[0,1].set_title("Visit Status")

# Follow-Up
v["Follow Up Required"].value_counts().plot(kind="pie", ax=axes[0,2],
    autopct="%1.1f%%", colors=["#5B8DB8","#D98EA0"], labels=["No","Yes"])
axes[0,2].set_title("Follow-Up Required")

# Visits per Year
yr = v.groupby("Visit Year").size()
yr.plot(kind="bar", ax=axes[1,0], color="#B5A0D0")
axes[1,0].set_title("Visits per Year"); axes[1,0].tick_params(axis="x", rotation=0)

# Visits per Month (overall)
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
mv = v["Visit MonthName"].value_counts().reindex(month_order, fill_value=0)
mv.plot(kind="bar", ax=axes[1,1], color="#5B8DB8")
axes[1,1].set_title("Visits by Month"); axes[1,1].tick_params(axis="x", rotation=45)

# Top 10 Diagnoses
top_dx = v["Diagnosis"].value_counts().head(10)
top_dx.plot(kind="barh", ax=axes[1,2], color="#A8C8A0")
axes[1,2].set_title("Top 10 Diagnoses")

plt.tight_layout()
plt.savefig("output_charts/02_visit_analysis.png", bbox_inches="tight", dpi=150)
plt.close()
print("  ✓ 02_visit_analysis.png")


# ── Figure 3 : Treatment & Cost ───────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Treatment & Cost Analysis", fontsize=16, fontweight="bold")

# Treatment Type
t["Treatment Type"].value_counts().plot(kind="bar", ax=axes[0,0], color="#D98EA0")
axes[0,0].set_title("Treatment Type Distribution")
axes[0,0].tick_params(axis="x", rotation=30)

# Treatment Status
t["Status"].value_counts().plot(kind="pie", ax=axes[0,1], autopct="%1.1f%%",
    colors=["#A8C8A0","#D98EA0","#F5C97A"])
axes[0,1].set_title("Treatment Status")

# Treatment Cost distribution
axes[1,0].hist(t["Treatment Cost"].dropna(), bins=40, color="#5B8DB8", edgecolor="white")
axes[1,0].set_title("Treatment Cost Distribution")
axes[1,0].set_xlabel("Cost ($)"); axes[1,0].set_ylabel("Frequency")

# Avg Cost by Treatment Type
avg_cost = t.groupby("Treatment Type")["Treatment Cost"].mean().sort_values()
avg_cost.plot(kind="barh", ax=axes[1,1], color="#B5A0D0")
axes[1,1].set_title("Avg Treatment Cost by Type")
axes[1,1].set_xlabel("Avg Cost ($)")

plt.tight_layout()
plt.savefig("output_charts/03_treatment_cost.png", bbox_inches="tight", dpi=150)
plt.close()
print("  ✓ 03_treatment_cost.png")


# ── Figure 4 : Lab Test Analysis ──────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Lab Test Analysis", fontsize=16, fontweight="bold")

l["Test Result"].value_counts().plot(kind="bar", ax=axes[0], color="#5B8DB8")
axes[0].set_title("Test Result Distribution"); axes[0].tick_params(axis="x", rotation=0)

l["Test Name"].value_counts().head(10).plot(kind="barh", ax=axes[1], color="#D98EA0")
axes[1].set_title("Top 10 Test Types")

l["Reference Range"].value_counts().plot(kind="pie", ax=axes[2],
    autopct="%1.1f%%", colors=["#A8C8A0","#D98EA0","#F5C97A"])
axes[2].set_title("Reference Range")

plt.tight_layout()
plt.savefig("output_charts/04_lab_test.png", bbox_inches="tight", dpi=150)
plt.close()
print("  ✓ 04_lab_test.png")


# ── Figure 5 : Doctor Workload ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Doctor & Specialty Analysis", fontsize=16, fontweight="bold")

spec = master.groupby("Specialty")["Visit ID"].count().sort_values(ascending=False)
spec.plot(kind="bar", ax=axes[0], color="#5B8DB8")
axes[0].set_title("Visits per Specialty"); axes[0].tick_params(axis="x", rotation=45)

top_docs = master.groupby("Doctor Name")["Visit ID"].count().sort_values().tail(10)
top_docs.plot(kind="barh", ax=axes[1], color="#B5A0D0")
axes[1].set_title("Top 10 Busiest Doctors")

plt.tight_layout()
plt.savefig("output_charts/05_doctor_workload.png", bbox_inches="tight", dpi=150)
plt.close()
print("  ✓ 05_doctor_workload.png")


4. GENERATING EDA CHARTS
  ✓ 01_patient_demographics.png
  ✓ 02_visit_analysis.png
  ✓ 03_treatment_cost.png
  ✓ 04_lab_test.png
  ✓ 05_doctor_workload.png


### Average Treatment Cost by Diagnosis

In [13]:
# ─────────────────────────────────────────────
# 5. KEY KPI SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. KEY KPIs")
print("=" * 60)

total_patients   = p["Patient ID"].nunique()
total_doctors    = d["Doctor ID"].nunique()
total_visits     = v["Visit ID"].nunique()
avg_age          = p["Age"].mean()
followup_rate    = (v["Follow Up Required"] == "Yes").mean() * 100
avg_treat_cost   = t["Treatment Cost"].mean()
total_lab_tests  = l["Lab Result ID"].nunique()
abnormal_rate    = (l["Test Result"] == "Abnormal").mean() * 100
avg_pts_per_doc  = total_visits / total_doctors
cancelled_rate   = (v["Visit Status"] == "Cancelled").mean() * 100
completed_treat  = (t["Status"] == "Completed").mean() * 100

kpis = {
    "Total Patients"               : f"{total_patients:,}",
    "Total Doctors"                : f"{total_doctors:,}",
    "Total Visits"                 : f"{total_visits:,}",
    "Avg Patient Age"              : f"{avg_age:.1f} yrs",
    "Follow-Up Rate"               : f"{followup_rate:.1f}%",
    "Avg Treatment Cost"           : f"${avg_treat_cost:,.2f}",
    "Total Lab Tests"              : f"{total_lab_tests:,}",
    "Abnormal Lab Result Rate"     : f"{abnormal_rate:.1f}%",
    "Avg Visits per Doctor"        : f"{avg_pts_per_doc:.1f}",
    "Cancelled Visit Rate"         : f"{cancelled_rate:.1f}%",
    "Treatment Completion Rate"    : f"{completed_treat:.1f}%",
}

for k, v_ in kpis.items():
    print(f"  {k:<35} {v_}")



5. KEY KPIs
  Total Patients                      10,000
  Total Doctors                       1,000
  Total Visits                        10,000
  Avg Patient Age                     49.8 yrs
  Follow-Up Rate                      49.8%
  Avg Treatment Cost                  $524.75
  Total Lab Tests                     10,000
  Abnormal Lab Result Rate            33.5%
  Avg Visits per Doctor               10.0
  Cancelled Visit Rate                33.4%
  Treatment Completion Rate           33.2%


In [15]:
# ─────────────────────────────────────────────
# 6. EXPORT CLEAN CSVs  (for SQL & Power BI)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("6. EXPORTING CLEAN CSVs")
print("=" * 60)

p.to_csv("output_clean/patient_clean.csv", index=False)
d.to_csv("output_clean/doctor_clean.csv", index=False)
v.to_csv("output_clean/visit_clean.csv", index=False)
t.to_csv("output_clean/treatment_clean.csv", index=False)
l.to_csv("output_clean/labtest_clean.csv", index=False)
master.to_csv("output_clean/master_table.csv", index=False)

for fname in ["patient_clean","doctor_clean","visit_clean","treatment_clean","labtest_clean","master_table"]:
    df_tmp = pd.read_csv(f"output_clean/{fname}.csv")
    print(f"  ✓ {fname}.csv  — {df_tmp.shape[0]:,} rows × {df_tmp.shape[1]} cols")

print("\n  All done! ")



6. EXPORTING CLEAN CSVs
  ✓ patient_clean.csv  — 10,000 rows × 23 cols
  ✓ doctor_clean.csv  — 1,000 rows × 8 cols
  ✓ visit_clean.csv  — 10,000 rows × 14 cols
  ✓ treatment_clean.csv  — 10,000 rows × 12 cols
  ✓ labtest_clean.csv  — 10,000 rows × 8 cols
  ✓ master_table.csv  — 10,000 rows × 37 cols

  All done! 


In [20]:
!pip install pymysql sqlalchemy
from sqlalchemy import create_engine
# MySQL connection
# IMPORTANT: Replace these placeholders with your actual remote MySQL database credentials
username = "root" # e.g., 'admin'
password = "your_password" # e.g., 'mysecretpassword'
host = "127.0.0.1"    # e.g., '192.168.1.100' or 'your.database.com'
port = "3306"                    # Default MySQL port, change if different
database = "paitent_healthcare"  # Your database name
engine = create_engine(f"mysql+pymysql://{username} : {password}@{host} :{port}/{database}")
# Write DataFrame to MySQL
table_name = "mytable" # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)
# Read back sample
pd.read_sql("SELECT * FROM mytable LIMIT 5;", engine)

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '127.0.0.1 ' ([Errno -2] Name or service not known)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)